# 07f — BCL POI 2008 acquisition / provenance / CRS audit

Goal: determine whether the Beijing City Lab 2008 POI source can be used reproducibly on Modal **before** any semantic join.

This notebook is gate-first:

- public Figshare metadata/file probe;
- explicit licence metadata check;
- cached/manual fallback on the Modal Volume;
- resumable large-file download;
- MDB/ZIP inspection in an isolated Modal worker with `mdbtools` + GDAL PGeo;
- CRS gate;
- no WORK/OFFICE/HOME or occupation inference.


In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='eda/07f-bcl-poi-2008-acquisition'
REPO_DIR=Path('/root/geolife')

if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())


In [ ]:
import pandas as pd
import httpx

spec=importlib.util.spec_from_file_location('s07f',REPO_DIR/'analysis'/'07f_bcl_poi_2008_acquisition.py')
s07f=importlib.util.module_from_spec(spec); sys.modules[spec.name]=s07f; spec.loader.exec_module(s07f)
print(s07f.synthetic_self_check())

V=Path('/mnt/geolife-data')
if not V.exists(): raise FileNotFoundError('Attach the GeoLife Modal Volume at /mnt/geolife-data')
C=V/'cache'/'07f_bcl_poi_2008_acquisition'; C.mkdir(parents=True,exist_ok=True)
EXT=V/'external'/'bcl_poi_2008'; EXT.mkdir(parents=True,exist_ok=True)
INBOX=EXT/'inbox'; INBOX.mkdir(parents=True,exist_ok=True)

anchor_paths=sorted(V.glob('**/anchor_observation_dates_private.pkl'))
if not anchor_paths: raise FileNotFoundError('corrected Stage 07c anchor_observation_dates_private.pkl is required')
anchors=pd.read_pickle(anchor_paths[0])
display(s07f.build_bcl_eligible_anchor_summary(anchors))


## 1. Public metadata / file / licence probe

Official record identifiers are pinned in the analysis module. Public Figshare metadata/files do not require authentication.


In [ ]:
candidate_pool=[p for p in INBOX.iterdir() if p.is_file() and p.suffix.lower() in {'.mdb','.zip'}]
raw_dir=EXT/'raw'
if raw_dir.exists(): candidate_pool += [p for p in raw_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.mdb','.zip'}]
priority={'.mdb':0,'.zip':1}
local_candidates=sorted(candidate_pool,key=lambda p:(priority.get(p.suffix.lower(),9),p.name,str(p)))
    metadata_status=int(response.status_code)
    if response.status_code==200:
        metadata=response.json()
    else:
        metadata_error=response.text[:500]
except Exception as exc:
    metadata_error=f'{type(exc).__name__}: {exc}'

metadata_summary=s07f.summarize_figshare_metadata(metadata,http_status=metadata_status,error=metadata_error)
files=s07f.figshare_files(metadata or {})
display(metadata_summary)
display(files)

candidate_pool=[*INBOX.glob('*.mdb'),*INBOX.glob('*.zip')]
if (EXT/'raw').exists(): candidate_pool += [*(EXT/'raw').glob('*.mdb'),*(EXT/'raw').glob('*.zip')]
priority={'.mdb':0,'.zip':1}
local_candidates=sorted(candidate_pool,key=lambda p:(priority.get(p.suffix.lower(),9),p.name,str(p)))
initial_decision=s07f.evaluate_acquisition_gates(metadata,http_status=metadata_status,error=metadata_error,local_candidate_files=[str(p) for p in local_candidates])
print('INITIAL GATES')
display(s07f.gates_frame(initial_decision))


## 2. Select acquisition path

Priority is deterministic: public Figshare MDB/ZIP if available; otherwise a manually uploaded/cached MDB/ZIP under `/mnt/geolife-data/external/bcl_poi_2008/inbox/`.

If neither exists, the notebook stays blocked and prints the exact inbox path instead of failing downstream.


In [ ]:
public_candidate=s07f.select_public_download_candidate(metadata or {})
source_mode=None; source_url=''; source_name=''; expected_size=None; expected_md5=''; manual_path=''
license_allows_public_download=initial_decision.license_status.startswith('pass_')

if public_candidate is not None and license_allows_public_download:
    source_mode='public_figshare'
    source_url=public_candidate['download_url']
    source_name=public_candidate['name']
    expected_size=public_candidate['size_bytes']
    expected_md5=public_candidate['computed_md5'] or public_candidate['supplied_md5']
elif local_candidates:
    source_mode='manual_or_cached'
    manual_path=str(local_candidates[0])
    source_name=Path(manual_path).name

print('source_mode:',source_mode)
print('source_name:',source_name or None)
if public_candidate is not None and not license_allows_public_download:
    print('PUBLIC FILE NOT AUTO-DOWNLOADED: explicit reusable licence gate did not pass.')
if source_mode is None:
    print('BLOCKED: no licence-cleared public MDB/ZIP and no manual/cached file.')
    print('Manual fallback inbox:',INBOX)


## 3. Modal acquisition + MDB inspection worker

The worker uses a dedicated Debian image with system dependencies. This avoids mutating the notebook kernel and keeps large-file work resumable on the persistent Volume.

If your Modal Volume is not named `geolife-data`, set `GEOLIFE_MODAL_VOLUME_NAME` before running this cell.


In [ ]:
worker_result=None
if source_mode is not None:
    import modal
    volume_name=os.environ.get('GEOLIFE_MODAL_VOLUME_NAME','geolife-data')
    data_volume=modal.Volume.from_name(volume_name,create_if_missing=False)
    app=modal.App('geolife-stage07f-bcl-poi-2008')
    worker_image=(modal.Image.debian_slim(python_version=f'{sys.version_info.major}.{sys.version_info.minor}')
        .apt_install('mdbtools','unixodbc','odbc-mdbtools','gdal-bin','unzip')
        .pip_install('httpx'))

    @app.function(image=worker_image,volumes={'/mnt/geolife-data':data_volume},timeout=1800,serialized=True)
    def acquire_and_inspect(payload: dict) -> dict:
        from hashlib import md5, sha256
        from pathlib import Path
        from urllib.parse import urlparse
        import json, os, shutil, subprocess, zipfile
        import httpx

        root=Path('/mnt/geolife-data')
        ext=root/'external'/'bcl_poi_2008'
        raw=ext/'raw'; raw.mkdir(parents=True,exist_ok=True)
        extracted=ext/'extracted'; extracted.mkdir(parents=True,exist_ok=True)

        def digest(path,algo):
            h=md5() if algo=='md5' else sha256()
            with Path(path).open('rb') as f:
                for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
            return h.hexdigest()

        def verify(path):
            p=Path(path)
            if not p.exists(): return {'exists':False}
            size=p.stat().st_size
            expected_size=payload.get('expected_size')
            expected_md5=(payload.get('expected_md5') or '').lower()
            actual_md5=digest(p,'md5')
            return {'exists':True,'size_bytes':size,'size_ok':expected_size in (None,'') or int(expected_size)==size,'md5':actual_md5,'md5_ok':not expected_md5 or actual_md5.lower()==expected_md5,'sha256':digest(p,'sha256')}

        if payload['source_mode']=='public_figshare':
            name=Path(payload.get('source_name') or urlparse(payload['source_url']).path).name or 'bcl_poi_2008.bin'
            dest=raw/name; part=raw/(name+'.part')
            current=verify(dest)
            if not (current.get('exists') and current.get('size_ok') and current.get('md5_ok')):
                start=part.stat().st_size if part.exists() else 0
                headers={'Range':f'bytes={start}-'} if start else {}
                with httpx.Client(follow_redirects=True,timeout=httpx.Timeout(60.0,read=300.0)) as client:
                    with client.stream('GET',payload['source_url'],headers=headers) as r:
                        r.raise_for_status()
                        mode='ab' if start and r.status_code==206 else 'wb'
                        if mode=='wb' and part.exists(): part.unlink()
                        with part.open(mode) as f:
                            for chunk in r.iter_bytes(chunk_size=8*1024*1024):
                                if chunk: f.write(chunk)
                part.replace(dest)
            source=dest
        else:
            source=Path(payload['manual_path'])
            try:
                source.resolve().relative_to(root.resolve())
            except Exception:
                return {'ok':False,'error':'manual path must be inside /mnt/geolife-data'}
            if not source.exists(): return {'ok':False,'error':f'manual file not visible in worker: {source}'}

        verification=verify(source)
        if not verification.get('exists') or not verification.get('size_ok',True) or not verification.get('md5_ok',True):
            return {'ok':False,'error':'download verification failed','source_path':str(source),'verification':verification}

        search_root=source.parent
        if source.suffix.lower()=='.zip':
            target=extracted/source.stem; target.mkdir(parents=True,exist_ok=True)
            with zipfile.ZipFile(source) as zf:
                base=target.resolve()
                for info in zf.infolist():
                    out=(target/info.filename).resolve()
                    if base!=out and base not in out.parents:
                        return {'ok':False,'error':f'unsafe archive path: {info.filename}'}
                zf.extractall(target)
            search_root=target

        mdbs=sorted([p for p in search_root.rglob('*') if p.is_file() and p.suffix.lower()=='.mdb'],key=lambda p:(-p.stat().st_size,str(p)))
        if source.suffix.lower()=='.mdb': mdbs=[source]
        if not mdbs: return {'ok':False,'error':'no .mdb found after acquisition','source_path':str(source),'verification':verification}
        mdb=mdbs[0]

        def run(cmd,timeout=900):
            try:
                cp=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout)
                return {'returncode':cp.returncode,'stdout':cp.stdout[:750000],'stderr':cp.stderr[:100000]}
            except Exception as exc:
                return {'returncode':-1,'stdout':'','stderr':f'{type(exc).__name__}: {exc}'}

        tables_run=run(['mdb-tables','-1',str(mdb)])
        schema_run=run(['mdb-schema',str(mdb),'postgres'])
        formats_run=run(['ogrinfo','--formats'])
        pgeo_available='PGeo' in formats_run['stdout']
        ogr_run=run(['ogrinfo','-ro','-so','-al',str(mdb)])
        if ogr_run['returncode']!=0 and pgeo_available:
            ogr_run=run(['ogrinfo','-ro','-so','-al','PGeo:'+str(mdb)])

        report={'ok':True,'source_path':str(source),'verification':verification,'mdb_path':str(mdb),'mdb_candidates':[str(p) for p in mdbs],'mdbtools_available':shutil.which('mdb-tables') is not None,'gdal_pgeo_available':pgeo_available,'mdb_tables_stdout':tables_run['stdout'],'mdb_tables_stderr':tables_run['stderr'],'mdb_schema_stdout':schema_run['stdout'],'mdb_schema_stderr':schema_run['stderr'],'ogr_stdout':ogr_run['stdout'],'ogr_stderr':ogr_run['stderr'],'ogr_returncode':ogr_run['returncode']}
        cache=root/'cache'/'07f_bcl_poi_2008_acquisition'; cache.mkdir(parents=True,exist_ok=True)
        (cache/'worker_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
        data_volume.commit()
        return report

    payload={'source_mode':source_mode,'source_url':source_url,'source_name':source_name,'expected_size':None if expected_size is None or pd.isna(expected_size) else int(expected_size),'expected_md5':expected_md5,'manual_path':manual_path}
    with app.run():
        worker_result=acquire_and_inspect.remote(payload)
    print('worker ok:',worker_result.get('ok'))
    print('worker error:',worker_result.get('error'))


## 4. Parse inspection report and freeze gates


In [ ]:
inspection_summary=pd.DataFrame(); final_decision=initial_decision; manifest={}
if worker_result and worker_result.get('ok'):
    parsed_ogr=s07f.parse_ogrinfo_report(worker_result.get('ogr_stdout',''))
    report={'mdb_path':worker_result.get('mdb_path'),'mdbtools_available':worker_result.get('mdbtools_available'),'gdal_pgeo_available':worker_result.get('gdal_pgeo_available'),'mdb_tables':s07f.parse_mdb_tables(worker_result.get('mdb_tables_stdout','')),'ogr':parsed_ogr,'error':worker_result.get('ogr_stderr','') if worker_result.get('ogr_returncode') else ''}
    inspection_summary=s07f.summarize_mdb_inspection(report)
    crs_text=s07f.choose_crs_text(report)
    structure_ok=s07f.mdb_structure_is_plausible(report)
    final_decision=s07f.evaluate_acquisition_gates(metadata,http_status=metadata_status,error=metadata_error,inspected_crs=crs_text,inspected_structure_ok=structure_ok,local_candidate_files=[worker_result.get('source_path','')])
    display(inspection_summary)
    print('OGR layers:',parsed_ogr.get('layers'))
    print('geometry types:',parsed_ogr.get('geometry_types'))
    print('feature counts:',parsed_ogr.get('feature_counts'))
    print('EPSG:',parsed_ogr.get('epsg_codes'))
    print('fields:',parsed_ogr.get('field_names'))

print('FINAL GATES')
display(s07f.gates_frame(final_decision))

if worker_result and worker_result.get('ok'):
    manifest=s07f.acquisition_manifest(source_mode=source_mode,source_url=source_url,local_path=worker_result.get('source_path',''),file_verification=worker_result.get('verification',{}),metadata_summary=metadata_summary,gate_decision=final_decision,inspection_summary=inspection_summary)
else:
    manifest=s07f.acquisition_manifest(source_mode=source_mode or 'none',source_url=source_url,local_path=manual_path,file_verification={},metadata_summary=metadata_summary,gate_decision=final_decision,inspection_summary=inspection_summary)

(C/'acquisition_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
metadata_summary.to_csv(C/'figshare_metadata_summary.csv',index=False)
files.to_csv(C/'figshare_files.csv',index=False)
s07f.gates_frame(final_decision).to_csv(C/'acquisition_gates.csv',index=False)
if not inspection_summary.empty: inspection_summary.to_csv(C/'mdb_inspection_summary.csv',index=False)
print('saved:',C)


## Decision boundary

Only `runner_status == ready_for_normalization` permits a later Stage 07g normalization/extraction patch.

Even then, BCL 2008 contains place names rather than a trusted category label. Any later semantic mapping must use a predeclared high-precision lexical rule set and must remain external evidence, not WORK/OFFICE ground truth.
